### While the solvation entropy functions of this library are written to be generalizable to pymatgen `Structure` objects, the intended purpose of this library is for working with the output of JDFTx calculations. In this tutorial we go through the steps of getting solvation entropies from the output of a JDFTx calculation

### For a JDFTx calculation, I reccomend creating a `StructureVolume` object from the path to the calculation directory or the outfile itself. So long as it is the only JDFTx calculation in that directory, this is the best spot for the `StructureVolume` to store its cache. The `StructureVolume` will use the JDFTx IO outputs module from pymatgen (v. >???) to create a `Structure` object from the final version of the structure in the out file.

In [3]:
from os import getcwd
from pathlib import Path
from pymatgen.io.jdftx.outputs import JDFTXOutfile
from JDFTxFreeNrg.volume import StructureVolume

# For our solvent
h2o_outfile_path = Path(getcwd()) / "data" / "H2O_opt" / "out"
h2o_sv = StructureVolume.from_outfile_path(h2o_outfile_path, method="MC")

# For our solute
h3o_outfile_path = Path(getcwd()) / "data" / "H3O_opt" / "out"
h3o_sv = StructureVolume.from_outfile_path(h3o_outfile_path, method="MC")

### Once we have our `StructureVolume`'s with safe separate cache directories, we can follow all the steps outlined in `getting_volumes.ipynb` and `solve_entropy.ipynb` to obtain the solvated free energy of our optimized hydronium

In [ ]:
from JDFTxFreeNrg.solv_entropy import get_vfree, get_solv_entropy_trans, get_solv_entropy_rot, get_standard_state_correction
from JDFTxFreeNrg.standard import get_entropy_trans, get_ideal_gas_vol, get_entropy_rot

T = 300.

water_vol = h2o_sv.get_volume(npoints=1e7)
water_vfree = get_vfree(water_vol, 55.5)
h3o_vol = h3o_sv.get_volume(npoints=1e7)

h3o_solv_tr_entrop = get_solv_entropy_trans(h3o_sv, h3o_vol, water_vol, water_vfree, T)
h3o_solv_rot_entrop = get_solv_entropy_rot(h3o_sv, h3o_vol, water_vfree, T)
h3o_solv_entropy = h3o_solv_tr_entrop + h3o_solv_rot_entrop + get_standard_state_correction(T)

